# NYC Jobs Dataset — Data Engineering Assessment

**Dataset:** NYC Open Data — City of New York Job Postings  
**Engine:** PySpark 2.4.5  
**Author:** Candidate Submission

---

## Notebook Structure
1. Environment Setup
2. Data Ingestion & Schema Exploration
3. Data Exploration & KPIs
4. Data Processing & Feature Engineering
5. Output Storage
6. Test Cases

---
## 1. Environment Setup

In [ ]:
import findspark
findspark.init()

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, DateType
from pyspark.sql.window import Window

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import re
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

print('All libraries imported successfully.')

In [ ]:
# Initialise Spark Session — connects to the Spark master defined in docker-compose.yml
spark = (
    SparkSession.builder
    .appName("pyspark-assesment")
    .master("spark://master:7077")   # Spark master URI from docker-compose
    .config("spark.executor.memory", "1g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version : {spark.version}")
print(f"App name      : {spark.sparkContext.appName}")

---
## 2. Data Ingestion & Schema Exploration

In [ ]:
# ---------------------------------------------------------------------------
# Read source CSV.  All columns arrive as strings — we cast numerics later.
# ---------------------------------------------------------------------------
RAW_PATH    = "/dataset/nyc-jobs.csv"
OUTPUT_PATH = "/dataset/nyc-jobs-processed.parquet"

df_raw = (
    spark.read
    .option("header", "true")
    .option("multiLine", "true")   # job descriptions span multiple lines
    .option("escape", '"')
    .csv(RAW_PATH)
)

print(f"Rows : {df_raw.count():,}")
print(f"Cols : {len(df_raw.columns)}")
df_raw.printSchema()

In [ ]:
# ── Column-level profile ────────────────────────────────────────────────────
# Classify each column as Numeric, Date, or Categorical and report nulls.

NUMERIC_COLS     = ["Salary Range From", "Salary Range To", "# Of Positions"]
DATE_COLS        = ["Posting Date", "Post Until", "Posting Updated", "Process Date"]
CATEGORICAL_COLS = [
    "Agency", "Posting Type", "Job Category", "Full-Time/Part-Time indicator",
    "Salary Frequency", "Level", "Civil Service Title"
]
TEXT_COLS = [
    "Job Description", "Minimum Qual Requirements",
    "Preferred Skills", "Additional Information"
]

total = df_raw.count()
profile_rows = []
for col in df_raw.columns:
    null_cnt = df_raw.filter(F.col(col).isNull() | (F.trim(F.col(col)) == "")).count()
    dtype = "Numeric" if col in NUMERIC_COLS else (
            "Date" if col in DATE_COLS else (
            "Text" if col in TEXT_COLS else "Categorical"))
    distinct = df_raw.select(col).distinct().count()
    profile_rows.append({"Column": col, "Type": dtype,
                          "Nulls": null_cnt, "Null%": f"{100*null_cnt/total:.1f}%",
                          "Distinct Values": distinct})

profile_df = pd.DataFrame(profile_rows)
print(profile_df.to_string(index=False))

### Observations
- **All 28 columns** arrive as strings; salary and position count columns need casting.
- `Job Category`, `Level`, `Preferred Skills` and `Post Until` have notable nulls.
- `Salary Frequency` has three values: `Annual`, `Hourly`, `Daily` — KPI comparisons must normalise to annual.
- Date columns contain full ISO-8601 timestamps; we extract date parts for trend analysis.

---
## 3. Data Exploration & KPIs

### KPI 1 — Top 10 Job Postings per Category

In [ ]:
def get_top_categories(df: DataFrame, n: int = 10) -> DataFrame:
    return (
        df.filter(F.col("Job Category").isNotNull() & (F.trim(F.col("Job Category")) != ""))
          .groupBy("Job Category")
          .agg(F.count("*").alias("postings"))
          .orderBy(F.desc("postings"))
          .limit(n)
    )

top_cats = get_top_categories(df_raw)
top_cats.show(truncate=False)

In [ ]:
# Visualise
top_cats_pd = top_cats.toPandas()

fig, ax = plt.subplots(figsize=(13, 5))
bars = ax.barh(top_cats_pd["Job Category"][::-1], top_cats_pd["postings"][::-1],
               color=sns.color_palette("Blues_d", len(top_cats_pd)))
ax.bar_label(bars, padding=4, fontsize=9)
ax.set_xlabel("Number of Postings")
ax.set_title("KPI 1 — Top 10 Job Categories by Number of Postings", fontweight="bold")
plt.tight_layout()
plt.show()

### KPI 2 — Salary Distribution per Job Category

In [ ]:
def get_salary_distribution(df: DataFrame) -> DataFrame:
    return (
        df.filter(
            (F.col("Salary Frequency") == "Annual") &
            F.col("Job Category").isNotNull() &
            (F.trim(F.col("Job Category")) != "")
        )
        .withColumn("salary_from", F.col("Salary Range From").cast(DoubleType()))
        .withColumn("salary_to",   F.col("Salary Range To").cast(DoubleType()))
        .withColumn("salary_mid",  (F.col("salary_from") + F.col("salary_to")) / 2)
        .filter(F.col("salary_mid") > 0)
        .groupBy("Job Category")
        .agg(
            F.round(F.avg("salary_mid"),  0).alias("avg_salary"),
            F.round(F.min("salary_from"), 0).alias("min_salary"),
            F.round(F.max("salary_to"),   0).alias("max_salary"),
            F.count("*").alias("postings")
        )
        .orderBy(F.desc("avg_salary"))
    )

salary_dist = get_salary_distribution(df_raw)
salary_dist.show(15, truncate=False)

In [ ]:
sal_pd = salary_dist.toPandas().head(12)

fig, ax = plt.subplots(figsize=(14, 5))
x = range(len(sal_pd))
ax.bar(x, sal_pd["max_salary"], label="Max Salary",  color="#4e9af1", alpha=0.6)
ax.bar(x, sal_pd["avg_salary"], label="Avg Salary",  color="#1f5fa6", alpha=0.9)
ax.bar(x, sal_pd["min_salary"], label="Min Salary",  color="#a8c7f0", alpha=0.7)
ax.set_xticks(list(x))
ax.set_xticklabels(
    [c[:30] for c in sal_pd["Job Category"]], rotation=35, ha="right", fontsize=8
)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v/1000:.0f}k"))
ax.set_title("KPI 2 — Salary Distribution per Job Category (Annual, Top 12)", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

### KPI 3 — Correlation: Higher Degree vs Salary

In [ ]:
# UDF to extract highest degree mentioned in Minimum Qual Requirements
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

DEGREE_ORDER = {"High School": 1, "Associate": 2, "Bachelors": 3, "Masters": 4, "PhD": 5}

@udf(StringType())
def extract_degree(qual_text: str) -> str:
    if not qual_text:
        return None
    t = qual_text.lower()
    if re.search(r'ph\.?d|doctorate', t):              return "PhD"
    if re.search(r"master'?s?", t):                    return "Masters"
    if re.search(r"bachelor'?s?|baccalaureate", t):    return "Bachelors"
    if re.search(r'associate', t):                     return "Associate"
    if re.search(r'high school|hs diploma|ged', t):    return "High School"
    return None

def get_degree_salary_correlation(df: DataFrame) -> DataFrame:
    """
    Compute average salary grouped by the highest degree required.
    Restricts to Annual salaries for a fair comparison.

    Returns
    -------
    DataFrame [degree, avg_salary, postings] ordered by degree level.
    """
    return (
        df.filter(F.col("Salary Frequency") == "Annual")
          .withColumn("salary_mid",
                      (F.col("Salary Range From").cast(DoubleType()) +
                       F.col("Salary Range To").cast(DoubleType())) / 2)
          .filter(F.col("salary_mid") > 0)
          .withColumn("degree", extract_degree(F.col("Minimum Qual Requirements")))
          .filter(F.col("degree").isNotNull())
          .groupBy("degree")
          .agg(
              F.round(F.avg("salary_mid"), 0).alias("avg_salary"),
              F.count("*").alias("postings")
          )
    )

deg_sal = get_degree_salary_correlation(df_raw)
deg_sal.show()

In [ ]:
deg_pd = deg_sal.toPandas()
deg_pd["order"] = deg_pd["degree"].map(DEGREE_ORDER)
deg_pd = deg_pd.sort_values("order")

fig, ax1 = plt.subplots(figsize=(9, 5))
color = "#1f5fa6"
ax1.bar(deg_pd["degree"], deg_pd["avg_salary"], color=color, alpha=0.8)
ax1.set_ylabel("Average Annual Salary ($)", color=color)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v/1000:.0f}k"))

ax2 = ax1.twinx()
ax2.plot(deg_pd["degree"], deg_pd["postings"], color="tomato", marker="o", linewidth=2, label="# Postings")
ax2.set_ylabel("Number of Postings", color="tomato")
ax2.legend(loc="upper left")

ax1.set_title("KPI 3 — Degree Level vs Average Annual Salary", fontweight="bold")
plt.tight_layout()
plt.show()

# Pearson correlation between degree rank and salary
deg_pd_complete = deg_pd.dropna(subset=["avg_salary", "order"])
corr, pval = stats.pearsonr(deg_pd_complete["order"], deg_pd_complete["avg_salary"])
print(f"Pearson r = {corr:.3f}, p-value = {pval:.4f}")
print("→ Strong positive correlation: higher degree ↔ higher salary." if corr > 0.7 else
      "→ Moderate correlation between degree and salary.")

### KPI 4 — Highest Salary Job Posting per Agency

In [ ]:
def get_highest_salary_per_agency(df: DataFrame) -> DataFrame:
    window = Window.partitionBy("Agency").orderBy(F.desc("salary_to"))

    return (
        df.filter(F.col("Salary Frequency") == "Annual")
          .withColumn("salary_to", F.col("Salary Range To").cast(DoubleType()))
          .filter(F.col("salary_to") > 0)
          .withColumn("rn", F.row_number().over(window))
          .filter(F.col("rn") == 1)
          .select(
              "Agency",
              "Business Title",
              F.col("salary_to").alias("max_salary_to")
          )
          .orderBy(F.desc("max_salary_to"))
    )

top_agency_jobs = get_highest_salary_per_agency(df_raw)
top_agency_jobs.show(15, truncate=False)

In [ ]:
ta_pd = top_agency_jobs.toPandas().head(15)

fig, ax = plt.subplots(figsize=(13, 6))
bars = ax.barh(
    ta_pd["Agency"][::-1],
    ta_pd["max_salary_to"][::-1],
    color=sns.color_palette("Greens_d", len(ta_pd))
)
ax.bar_label(bars, labels=[f"${v/1000:.0f}k" for v in ta_pd["max_salary_to"][::-1]], padding=4, fontsize=8)
ax.set_xlabel("Highest Salary Range To ($)")
ax.set_title("KPI 4 — Highest Salary Job Posting per Agency (Top 15)", fontweight="bold")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v/1000:.0f}k"))
plt.tight_layout()
plt.show()

### KPI 5 — Average Salary per Agency (Last 2 Years)

In [ ]:
def get_avg_salary_last_n_years(df: DataFrame, n: int = 2) -> DataFrame:
    # Derive cutoff dynamically from the latest posting date in the dataset
    max_date_row = (
        df.withColumn("posting_date", F.to_date(F.col("Posting Date")))
          .agg(F.max("posting_date").alias("max_date"))
          .collect()[0]
    )
    max_date = max_date_row["max_date"]
    cutoff   = max_date - pd.DateOffset(years=n)  # Python fallback for simplicity

    return (
        df.filter(F.col("Salary Frequency") == "Annual")
          .withColumn("posting_date", F.to_date(F.col("Posting Date")))
          .filter(F.col("posting_date") >= F.lit(str(cutoff.date())))
          .withColumn("salary_mid",
                      (F.col("Salary Range From").cast(DoubleType()) +
                       F.col("Salary Range To").cast(DoubleType())) / 2)
          .filter(F.col("salary_mid") > 0)
          .groupBy("Agency")
          .agg(
              F.round(F.avg("salary_mid"), 0).alias("avg_salary"),
              F.count("*").alias("postings")
          )
          .orderBy(F.desc("avg_salary"))
    )

avg_agency = get_avg_salary_last_n_years(df_raw, n=2)
avg_agency.show(15, truncate=False)

In [ ]:
aa_pd = avg_agency.toPandas().head(15)

fig, ax = plt.subplots(figsize=(13, 6))
bars = ax.barh(
    aa_pd["Agency"][::-1],
    aa_pd["avg_salary"][::-1],
    color=sns.color_palette("Purples_d", len(aa_pd))
)
ax.bar_label(bars, labels=[f"${v/1000:.0f}k" for v in aa_pd["avg_salary"][::-1]], padding=4, fontsize=8)
ax.set_xlabel("Average Annual Salary ($)")
ax.set_title("KPI 5 — Avg Salary per Agency (Last 2 Years, Top 15)", fontweight="bold")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v/1000:.0f}k"))
plt.tight_layout()
plt.show()

### KPI 6 — Highest Paid Skills in the US Market (from Preferred Skills field)

In [ ]:
# Known high-value skills to search for in the Preferred Skills column
SKILLS = [
    "Python", "SQL", "Java", "Spark", "Tableau", "Power BI",
    "Machine Learning", "Excel", "R ", "GIS", "AWS", "Azure",
    "Hadoop", "Scala", "Kubernetes", "Docker", "AutoCAD"
]

def get_top_paid_skills(df: DataFrame, skills: list) -> pd.DataFrame:
    annual = (
        df.filter(
            (F.col("Salary Frequency") == "Annual") &
            F.col("Preferred Skills").isNotNull()
        )
        .withColumn("salary_mid",
                    (F.col("Salary Range From").cast(DoubleType()) +
                     F.col("Salary Range To").cast(DoubleType())) / 2)
        .filter(F.col("salary_mid") > 0)
        .withColumn("skills_lower", F.lower(F.col("Preferred Skills")))
    )

    results = []
    for skill in skills:
        filtered = annual.filter(F.col("skills_lower").contains(skill.lower()))
        cnt = filtered.count()
        if cnt > 0:
            avg = filtered.agg(F.avg("salary_mid")).collect()[0][0]
            results.append({"skill": skill.strip(), "avg_salary": round(avg), "postings": cnt})

    return pd.DataFrame(results).sort_values("avg_salary", ascending=False)

skills_pd = get_top_paid_skills(df_raw, SKILLS)
print(skills_pd.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
colors = sns.color_palette("rocket", len(skills_pd))
bars = ax.barh(skills_pd["skill"][::-1], skills_pd["avg_salary"][::-1], color=colors)
ax.bar_label(bars, labels=[f"${v/1000:.0f}k" for v in skills_pd["avg_salary"][::-1]], padding=4, fontsize=9)
ax.set_xlabel("Average Annual Salary ($)")
ax.set_title("KPI 6 — Highest Paid Skills (NYC Govt Market)", fontweight="bold")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v/1000:.0f}k"))
plt.tight_layout()
plt.show()

---
## 4. Data Processing & Feature Engineering

### 4a. Cleaning Functions

In [ ]:
def cast_numeric_columns(df: DataFrame) -> DataFrame:
    """
    Cast salary and position-count columns to their correct numeric types.
    Invalid values are silently coerced to null by Spark.
    """
    return (
        df.withColumn("Salary Range From", F.col("Salary Range From").cast(DoubleType()))
          .withColumn("Salary Range To",   F.col("Salary Range To").cast(DoubleType()))
          .withColumn("# Of Positions",    F.col("# Of Positions").cast(IntegerType()))
    )


def cast_date_columns(df: DataFrame) -> DataFrame:
    """Parse ISO-8601 timestamp columns into proper DateType fields."""
    for col in ["Posting Date", "Post Until", "Posting Updated", "Process Date"]:
        df = df.withColumn(col, F.to_date(F.col(col)))
    return df


def drop_low_value_columns(df: DataFrame) -> DataFrame:
    """
    Remove columns that are either largely empty or provide no analytical
    signal: free-text admin fields and duplicate location columns.
    """
    cols_to_drop = [
        "To Apply",            # Admin instructions, not analytical
        "Additional Information",  # Unstructured, >80% redundant
        "Recruitment Contact", # Personal contact info
        "Hours/Shift",         # Very high null rate
        "Work Location 1",     # Duplicated by Work Location
    ]
    existing = [c for c in cols_to_drop if c in df.columns]
    return df.drop(*existing)


def fill_categorical_nulls(df: DataFrame) -> DataFrame:
    """Impute null categorical values with 'Unknown' to prevent row loss."""
    return (
        df.fillna({"Job Category": "Unknown",
                   "Level":        "Unknown",
                   "Preferred Skills": ""})
    )


print("Cleaning functions defined.")

### 4b. Feature Engineering

In [ ]:
# ── Feature 1: Annual Salary Midpoint ───────────────────────────────────────
def add_salary_midpoint(df: DataFrame) -> DataFrame:
    return (
        df.withColumn("salary_mid_raw",
                      (F.col("Salary Range From") + F.col("Salary Range To")) / 2)
          .withColumn("annual_salary_mid",
                      F.when(F.col("Salary Frequency") == "Annual",  F.col("salary_mid_raw"))
                       .when(F.col("Salary Frequency") == "Hourly",  F.col("salary_mid_raw") * 2080)
                       .when(F.col("Salary Frequency") == "Daily",   F.col("salary_mid_raw") * 260)
                       .otherwise(None))
          .drop("salary_mid_raw")
    )


# ── Feature 2: Salary Band / Tier ───────────────────────────────────────────
def add_salary_band(df: DataFrame) -> DataFrame:
    return df.withColumn(
        "salary_band",
        F.when(F.col("annual_salary_mid") <  50_000, "Entry")
         .when(F.col("annual_salary_mid") <  90_000, "Mid")
         .when(F.col("annual_salary_mid") < 130_000, "Senior")
         .otherwise("Executive")
    )


# ── Feature 3: Posting Age in Days ──────────────────────────────────────────
def add_posting_age(df: DataFrame) -> DataFrame:
    return (
        df.withColumn("posting_age_days",
                      F.datediff(F.col("Process Date"), F.col("Posting Date")))
          .withColumn("posting_age_days",
                      F.when(F.col("posting_age_days") >= 0, F.col("posting_age_days")).otherwise(None))
    )


# ── Feature 4: Posting Year / Quarter ───────────────────────────────────────
def add_time_features(df: DataFrame) -> DataFrame:
    """
    Feature Engineering #4 — Temporal Decomposition

    Extracts year and quarter from the Posting Date for trend analysis.
    """
    return (
        df.withColumn("posting_year",    F.year(F.col("Posting Date")))
          .withColumn("posting_quarter", F.quarter(F.col("Posting Date")))
    )


# ── Feature 5: Degree Level (ordinal) ───────────────────────────────────────
def add_degree_level(df: DataFrame) -> DataFrame:
    """
    Feature Engineering #5 — Minimum Degree Required

    Applies the extract_degree UDF (defined above) to create an ordinal
    degree column, then maps it to a numeric rank for correlation analysis.
    """
    degree_map = F.create_map(
        F.lit("High School"), F.lit(1),
        F.lit("Associate"),   F.lit(2),
        F.lit("Bachelors"),   F.lit(3),
        F.lit("Masters"),     F.lit(4),
        F.lit("PhD"),         F.lit(5),
    )
    return (
        df.withColumn("min_degree", extract_degree(F.col("Minimum Qual Requirements")))
          .withColumn("degree_rank", degree_map[F.col("min_degree")])
    )


print("Feature engineering functions defined.")

### 4c. Full Processing Pipeline

In [ ]:
def build_processed_dataset(df: DataFrame) -> DataFrame:
   
    return (
        df
        .transform(cast_numeric_columns)
        .transform(cast_date_columns)
        .transform(drop_low_value_columns)
        .transform(fill_categorical_nulls)
        .transform(add_salary_midpoint)
        .transform(add_salary_band)
        .transform(add_posting_age)
        .transform(add_time_features)
        .transform(add_degree_level)
    )

df_processed = build_processed_dataset(df_raw)
print(f"Processed rows : {df_processed.count():,}")
print(f"Processed cols : {len(df_processed.columns)}")
df_processed.printSchema()

In [ ]:
# Quick sanity check on the new engineered columns
df_processed.select(
    "Business Title", "annual_salary_mid", "salary_band",
    "posting_age_days", "posting_year", "min_degree", "degree_rank"
).show(10, truncate=False)

### 4d. Salary Band Distribution (Post Feature Engineering)

In [ ]:
band_pd = (
    df_processed.groupBy("salary_band")
                .count()
                .toPandas()
                .set_index("salary_band")
                .reindex(["Entry", "Mid", "Senior", "Executive"])
                .dropna()
)

fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(band_pd["count"], labels=band_pd.index, autopct="%1.1f%%",
       colors=sns.color_palette("Set2"), startangle=140)
ax.set_title("Salary Band Distribution across all Postings", fontweight="bold")
plt.tight_layout()
plt.show()

---
## 5. Store Processed Data

In [ ]:
def store_processed_data(df: DataFrame, path: str, fmt: str = "parquet") -> None:
  
    writer = df.coalesce(1).write.mode("overwrite")
    if fmt == "parquet":
        writer.parquet(path)
    elif fmt == "csv":
        writer.option("header", "true").csv(path)
    else:
        raise ValueError(f"Unsupported format: {fmt}")
    print(f"Data written to {path} ({fmt})")


store_processed_data(df_processed, OUTPUT_PATH, fmt="parquet")

# Verify round-trip
df_verify = spark.read.parquet(OUTPUT_PATH)
print(f"Verification read rows: {df_verify.count():,}")

---
## 6. Test Cases

In [ ]:
# ── Helper: assert with pass/fail print ─────────────────────────────────────
def run_test(name: str, actual, expected):
    try:
        assert actual == expected, f"Expected {expected!r}, got {actual!r}"
        print(f"  ✓  PASS  {name}")
    except AssertionError as e:
        print(f"  ✗  FAIL  {name} — {e}")


print("=" * 60)
print(" Running Test Suite")
print("=" * 60)

In [ ]:
# ── Test 1: get_salary_frequency (from starter notebook) ────────────────────
def get_salary_frequency(df: DataFrame) -> list:
    """Return distinct Salary Frequency values."""
    row_list = df.select('Salary Frequency').distinct().collect()
    return sorted([row['Salary Frequency'] for row in row_list])

def test_get_salary_frequency():
    mock_data = [('A', 'Annual'), ('B', 'Daily'), ('C', 'Annual')]
    schema    = ['id', 'Salary Frequency']
    mock_df   = spark.createDataFrame(data=mock_data, schema=schema)
    expected  = ['Annual', 'Daily']   # sorted, deduplicated
    run_test("get_salary_frequency — distinct, sorted",
             get_salary_frequency(mock_df), expected)

test_get_salary_frequency()

In [ ]:
# ── Test 2: get_top_categories ───────────────────────────────────────────────
def test_get_top_categories():
    mock_data = [
        ("Technology",), ("Technology",), ("Technology",),
        ("Legal",),      ("Legal",),
        ("Health",),
        (None,)
    ]
    schema  = ["Job Category"]
    mock_df = spark.createDataFrame(data=mock_data, schema=schema)
    result  = get_top_categories(mock_df, n=2).collect()

    run_test("get_top_categories — top 1 is Technology",
             result[0]["Job Category"], "Technology")
    run_test("get_top_categories — returns exactly n=2 rows",
             len(result), 2)
    run_test("get_top_categories — null rows excluded",
             all(r["Job Category"] is not None for r in result), True)

test_get_top_categories()

In [ ]:
# ── Test 3: add_salary_midpoint ──────────────────────────────────────────────
def test_add_salary_midpoint():
    schema = ["Salary Range From", "Salary Range To", "Salary Frequency"]
    mock_data = [
        ("60000", "80000", "Annual"),  # mid = 70 000
        ("20",   "30",    "Hourly"),   # mid = 25 × 2080 = 52 000
        ("200",  "300",   "Daily"),    # mid = 250 × 260 = 65 000
        ("0",    "0",     "Annual"),   # mid = 0
    ]
    mock_df = spark.createDataFrame(data=mock_data, schema=schema)
    result_df = (
        mock_df
        .withColumn("Salary Range From", F.col("Salary Range From").cast(DoubleType()))
        .withColumn("Salary Range To",   F.col("Salary Range To").cast(DoubleType()))
        .transform(add_salary_midpoint)
    )
    rows = result_df.select("annual_salary_mid").collect()

    run_test("add_salary_midpoint — Annual",  rows[0][0], 70_000.0)
    run_test("add_salary_midpoint — Hourly",  rows[1][0], 52_000.0)
    run_test("add_salary_midpoint — Daily",   rows[2][0], 65_000.0)

test_add_salary_midpoint()

In [ ]:
# ── Test 4: add_salary_band ──────────────────────────────────────────────────
def test_add_salary_band():
    schema   = ["annual_salary_mid"]
    mock_df  = spark.createDataFrame(
        [(30_000.0,), (70_000.0,), (110_000.0,), (150_000.0,)], schema
    )
    rows = mock_df.transform(add_salary_band).select("salary_band").collect()

    run_test("add_salary_band — Entry",     rows[0][0], "Entry")
    run_test("add_salary_band — Mid",       rows[1][0], "Mid")
    run_test("add_salary_band — Senior",    rows[2][0], "Senior")
    run_test("add_salary_band — Executive", rows[3][0], "Executive")

test_add_salary_band()

In [ ]:
# ── Test 5: extract_degree UDF ───────────────────────────────────────────────
def test_extract_degree():
    schema  = ["Minimum Qual Requirements"]
    mock_df = spark.createDataFrame([
        ("Requires a PhD in Computer Science",),
        ("Master's degree preferred",),
        ("Baccalaureate from an accredited college",),
        ("High school diploma required",),
        ("No formal requirement",),
    ], schema)
    rows = (
        mock_df.withColumn("degree", extract_degree(F.col("Minimum Qual Requirements")))
               .select("degree")
               .collect()
    )
    run_test("extract_degree — PhD",         rows[0][0], "PhD")
    run_test("extract_degree — Masters",     rows[1][0], "Masters")
    run_test("extract_degree — Bachelors",   rows[2][0], "Bachelors")
    run_test("extract_degree — High School", rows[3][0], "High School")
    run_test("extract_degree — None",        rows[4][0], None)

test_extract_degree()

print("=" * 60)
print(" All tests complete.")
print("=" * 60)

---
## Summary & Deployment Notes

### What was built
| Section | Output |
|---------|--------|
| Data Exploration | Column profile table, 6 KPI analyses with visualisations |
| Data Processing | 5-step cleaning pipeline + 5 feature engineering columns |
| Storage | Parquet output at `/dataset/nyc-jobs-processed.parquet` |
| Tests | 5 test functions covering all major logic |

### Triggering / Scheduling
- **Ad-hoc**: run this notebook via the Jupyter UI (`Run All`).
- **Scheduled**: export the pipeline functions to a `.py` script and submit via `spark-submit`:
  ```bash
  spark-submit --master spark://master:7077 /app/nyc_jobs_pipeline.py
  ```
- **Orchestration**: wrap the `spark-submit` call in an **Apache Airflow DAG** (or cron) for recurring execution on new data drops.

### Deployment Steps
1. Place updated CSV in `/dataset/nyc-jobs.csv` (or configure S3/ADLS path).
2. Run `docker compose -f docker-compose.yml --project-name my_assesment up -d`.
3. Submit the pipeline: `spark-submit --master spark://master:7077 /app/pipeline.py`.
4. Processed Parquet lands in `/dataset/nyc-jobs-processed.parquet` — consumable by any BI tool or downstream model.

### Assumptions & Limitations
- Salary comparisons limited to **Annual** postings for fairness; Hourly/Daily are normalised using standard working-year constants (2 080 h / 260 d).
- `"Preferred Skills"` column is free-text; skill extraction uses keyword matching — NLP-based extraction would improve KPI 6 precision.
- The dataset reflects historical City of NY postings; KPI 6 (market skills) is therefore specific to the public-sector NYC market, not the broader US market.
- `Post Until` has a high null rate — open-ended postings were excluded from any tenure analysis.